# WordEmbedding 기반 텍스트 분석 연습
> [02_text_analysis.ipynb]의 develop

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprocessing.ipynb]
2. Word Embedding model을 이용하여 벡터화한다.
3. 입력된 문자열의 긍/부정을 판단한다. (유사도 활용)

In [1]:
# # 한국어 혐오 데이터셋
# from Korpora import Korpora
# Korpora.fetch("korean_hate_speech")

In [2]:
import pandas as pd
from pathlib import Path

base_dir = Path("C:/Users/Playdata/Korpora/korean_hate_speech")

all_comments = []

# labeled
for path in [
    base_dir / "labeled" / "train.tsv",
    base_dir / "labeled" / "dev.tsv"
]:
    if path.exists():
        df = pd.read_csv(path, sep="\t")
        if "comments" in df.columns:
            all_comments.extend(df["comments"].dropna().astype(str).tolist())

# unlabeled 폴더 전체 읽기
unlabeled_dir = base_dir / "unlabeled"

for file_path in unlabeled_dir.glob("*"):
    try:
        if file_path.suffix == ".tsv":
            df = pd.read_csv(file_path, sep="\t")
            if "comments" in df.columns:
                all_comments.extend(df["comments"].dropna().astype(str).tolist())
        elif file_path.suffix == ".csv":
            df = pd.read_csv(file_path)
            if "comments" in df.columns:
                all_comments.extend(df["comments"].dropna().astype(str).tolist())
        elif file_path.suffix == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_comments.append(line)
    except Exception as e:
        print(f"읽기 실패: {file_path} / {e}")

# DataFrame으로 정리
corpus_df = pd.DataFrame({"comments": all_comments})

corpus_df = corpus_df.dropna()
corpus_df["comments"] = corpus_df["comments"].astype(str).str.strip()
corpus_df = corpus_df[corpus_df["comments"] != ""]
corpus_df = corpus_df.drop_duplicates()

print(corpus_df.shape) # 1850053개로 너무 많음

# 2만개 랜덤 추출로 데이터 양 줄이기
corpus_df = corpus_df.sample(n=20000, random_state=42)
print(corpus_df.shape)
print(corpus_df.head())

(1850053, 1)
(20000, 1)
                                                  comments
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...
562126                                         이가은 떨어지나보네욤
1080472                                 세기의결혼???아무거나 갔다붙이네
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요
995798                 사이다는 없는데요....뭐 마지막회에 몰아서 다 하려나...답답


In [3]:
import re

def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)       # 공백 정리
    text = re.sub(r"[^\w\s가-힣ㄱ-ㅎㅏ-ㅣ]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus_df["clean_comments"] = corpus_df["comments"].apply(clean_text)

# 너무 짧은 문장 제거
corpus_df = corpus_df[corpus_df["clean_comments"].str.len() >= 2]

print(corpus_df[["comments", "clean_comments"]].head())

                                                  comments  \
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...   
562126                                         이가은 떨어지나보네욤   
1080472                                 세기의결혼???아무거나 갔다붙이네   
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요   
995798                 사이다는 없는데요....뭐 마지막회에 몰아서 다 하려나...답답   

                                            clean_comments  
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...  
562126                                         이가은 떨어지나보네욤  
1080472                                   세기의결혼 아무거나 갔다붙이네  
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요  
995798                      사이다는 없는데요 뭐 마지막회에 몰아서 다 하려나 답답  


### 토크나이징 하기

In [ ]:
from konlpy.tag import Okt

okt = Okt() # 형태소 분석기 객체 생성

# 한국어 불용어 파일
with open('./etc/ko_stopwords.txt', 'r', encoding='UTF-8') as f:
    ko_stopwords =  [line.strip() for line in f]

tokenized_corpus = []

for text in corpus_df["clean_comments"]:
    tokens = okt.morphs(text, stem=True)
    tokens = [t for t in tokens if len(t) > 1]  # 한 글자 제거는 선택
    tokens = [token for token in tokens if token not in ko_stopwords]
    if tokens:
        tokenized_corpus.append(tokens)

print(tokenized_corpus[:3])
print(len(tokenized_corpus))

FileNotFoundError: [Errno 2] No such file or directory: 'ko_stopwords.txt'

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1
)

In [ ]:
model.wv.most_similar('행복',topn=100)

In [ ]:
model.wv.most_similar('불행',topn=100)